# CFM56-5B — Gázkar Szimulátor & ECAM Panel
**Interactive throttle control** with A320-style ECAM engine display.

- **Throttle 0%** → idle (~1000 K T4)
- **Throttle 100%** → TOGA — full thrust (~1700 K T4)

*N1/N2 are estimated from empirical correlations. EGT, FF, OPR, Thrust are directly from the thermodynamic simulation.*

In [1]:
import sys
sys.path.insert(0, '..')

import importlib
import visualization.station_diagram as sd
import visualization.ts_diagram as ts
import visualization.model_3d as m3d
importlib.reload(sd)
importlib.reload(ts)
importlib.reload(m3d)

from engine import run_design_point
from visualization.station_diagram import plot_station_diagram
from visualization.ts_diagram import plot_ts_diagram
from visualization.model_3d import plot_3d_model
from IPython.display import display
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
from IPython.display import HTML

FLIGHT_PHASES = {
    'Takeoff   (0 ft, Mach 0.25)':        {'altitude_ft': 0,     'mach': 0.25, 'key': 'takeoff'},
    'Climb     (15 000 ft, Mach 0.50)':   {'altitude_ft': 15000, 'mach': 0.50, 'key': 'climb'},
    'Cruise    (35 000 ft, Mach 0.78)':   {'altitude_ft': 35000, 'mach': 0.78, 'key': 'cruise'},
}

# ── Empirical N1/N2 correlations for CFM56-5B ────────────────────────────
# N1 (fan speed): ~20% at idle, ~100% at TOGA  → linear with throttle %
# N2 (core speed): ~55% at idle, ~100% at TOGA → linear with throttle %
# Source: CFM56-5B AMM / Aircraft Commerce issue 58
def estimate_n1(throttle_pct):
    return 20.0 + 0.80 * throttle_pct   # 20% → 100%

def estimate_n2(throttle_pct):
    return 55.0 + 0.45 * throttle_pct   # 55% → 100%

def ecam_html(result, n1, n2):
    """Generate A320-style ECAM engine panel as HTML."""
    egt = result.stations.get('S5_lpt_exit', result.stations.get('lpt_exit'))
    egt_c = f"{egt.T - 273.15:.0f}" if egt else "---"
    ff_kgh = f"{result.fuel_flow * 3600:.0f}"
    thr    = f"{result.thrust_kN:.1f}"
    opr    = f"{result.opr:.2f}"
    sfc    = f"{result.sfc:.5f}"
    n1s    = f"{n1:.1f}"
    n2s    = f"{n2:.1f}"

    return f"""
    <div style="
        background:#000; color:#00e000; font-family:'Courier New',monospace;
        border:2px solid #444; border-radius:6px; padding:16px 24px;
        display:inline-block; min-width:380px; margin:8px 0;">

      <div style="text-align:center; color:#00aaff; font-size:13px;
                  letter-spacing:3px; border-bottom:1px solid #333; padding-bottom:6px; margin-bottom:10px;">
        ── ENGINE 1 ──&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;── ENGINE 2 ──
      </div>

      <!-- N1 -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">N1</span>
        <span style="font-size:26px; font-weight:bold; color:#00ff00;">{n1s}</span>
        <span style="font-size:26px; font-weight:bold; color:#00ff00;">{n1s}</span>
        <span style="color:#aaa; font-size:11px;">%</span>
      </div>

      <!-- EGT -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">EGT</span>
        <span style="font-size:22px; color:#ffaa00;">{egt_c}</span>
        <span style="font-size:22px; color:#ffaa00;">{egt_c}</span>
        <span style="color:#aaa; font-size:11px;">°C</span>
      </div>

      <!-- N2 -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">N2</span>
        <span style="font-size:18px; color:#00cc00;">{n2s}</span>
        <span style="font-size:18px; color:#00cc00;">{n2s}</span>
        <span style="color:#aaa; font-size:11px;">%</span>
      </div>

      <div style="border-top:1px solid #333; margin:8px 0;"></div>

      <!-- FF -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">FF</span>
        <span style="font-size:18px; color:#00e000;">{ff_kgh}</span>
        <span style="font-size:18px; color:#00e000;">{ff_kgh}</span>
        <span style="color:#aaa; font-size:11px;">KG/H</span>
      </div>

      <!-- THR -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">THR</span>
        <span style="font-size:18px; color:#00e000;">{thr}</span>
        <span style="font-size:18px; color:#00e000;">{thr}</span>
        <span style="color:#aaa; font-size:11px;">kN</span>
      </div>

      <!-- OPR -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">OPR</span>
        <span style="font-size:18px; color:#00e000;">{opr}</span>
        <span style="font-size:18px; color:#00e000;">{opr}</span>
        <span style="color:#aaa; font-size:11px;"></span>
      </div>

      <!-- SFC -->
      <div style="display:flex; justify-content:space-between; align-items:baseline; margin:4px 0;">
        <span style="color:#aaa; font-size:11px;">SFC</span>
        <span style="font-size:14px; color:#00cc00;">{sfc}</span>
        <span style="font-size:14px; color:#00cc00;">{sfc}</span>
        <span style="color:#aaa; font-size:11px;">kg/kN·s</span>
      </div>

      <div style="text-align:center; color:#555; font-size:9px; margin-top:10px;">
        N1/N2 ESTIMATED · EGT/FF/OPR/THR FROM SIMULATION
      </div>
    </div>
    """

# ── Widgets ───────────────────────────────────────────────────────────────
throttle_slider = widgets.IntSlider(
    value=100, min=0, max=100, step=5,
    description='Throttle [%]:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='420px')
)

phase_dropdown = widgets.Dropdown(
    options=list(FLIGHT_PHASES.keys()),
    value=list(FLIGHT_PHASES.keys())[0],
    description='Flight phase:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='420px')
)

controls = widgets.HBox([phase_dropdown, throttle_slider])
output   = widgets.Output()

def update(change):
    pct = throttle_slider.value / 100.0
    T4  = 1000.0 + pct * 700.0
    fp  = FLIGHT_PHASES[phase_dropdown.value]
    n1  = estimate_n1(throttle_slider.value)
    n2  = estimate_n2(throttle_slider.value)

    result = run_design_point(
        flight_phase=fp['key'],
        altitude_ft=fp['altitude_ft'],
        mach=fp['mach'],
        T4_override=T4,
    )

    with output:
        output.clear_output(wait=True)

        # ECAM panel
        display(HTML(ecam_html(result, n1, n2)))

        # Station diagram
        fig1 = plot_station_diagram(result)
        plt.tight_layout()
        display(fig1)
        plt.close(fig1)

        # T-s diagram
        fig2 = plot_ts_diagram([result])
        plt.tight_layout()
        display(fig2)
        plt.close(fig2)

        # 3D model
        fig3 = plot_3d_model(result)
        display(fig3)

throttle_slider.observe(update, names='value')
phase_dropdown.observe(update, names='value')

display(controls, output)
update(None)